# Advanced Concepts in Docker Compose

We have a working multi-container pipeline running through Docker Compose. You can start your services with a single command, connect a Python ETL script to a Postgres database, and even persist your data across runs. 

Now, we are going to move from local development to a production use. That includes adding health checks to prevent race conditions, using multi-stage Docker builds to reduce image size and complexity, running as a non-root user to improve security, and externalizing secrets with environment files.

We will focused on how it runs, how it’s configured, and how ready it is for the environments where it might eventually live.

To accomplish this we are going to perform the following steps:
1. Added a health check to make sure services only start when they’re truly ready.
2. Rewrote your Dockerfile using a multi-stage build, slimming down your image and separating build concerns from runtime needs.
3. Hardened your container by running it as a non-root user and moved configuration into a .env file to make it safer and more shareable.

These are the kinds of improvements developers make every day when preparing pipelines for staging, production, or CI. Whether you’re working in Docker, Kubernetes, or a cloud platform, these patterns are part of the job.

## Add a Health Check to the Database

At this point, we have a two services defined in our `docker-compose.yaml` which are the PostreSQL databse and the Python container that runs an ETL script. Both services start together, and your script connects to the database over the shared Compose network.

That setup works, but it has a hidden risk. When you run docker compose up, Docker starts each container, but **it doesn’t check whether those services are actually ready**. If Postgres takes a few seconds to initialize, your app might try to connect too early and either fail or hang without a clear explanation.

Here, you can find a [Docker Health Check: A Practical Guide](https://www.dash0.com/guides/docker-health-check-a-practical-guide). 

In order to check the services are ready, we need to update our `docker-compose.yaml`. First, I will show to you what we will add and afterwards, I will explain what it does and means. here, you can find the code:

```
services:
    ...
    db:
        image: postgres:15
        healthcheck:
            test: ["CMD", "pg_isready", "-U", "postgres"]
            interval: 10s
            timeout: 2s
            retries: 5
            start_period: 3s
            ...
```

To active this health check, we need to add `healthcheck` inside the database, which will monitor the readiness of the Postgres container. This gives Docker an explicit test to run, rather than relying on the assumption that "started" means "ready." 

- test: Run the command to make the health check. `CMD pg_isready -U postgres` in which pg_isready is a built-in command from Postgres which make the check easier to make.
- interval: Maximum time interval to check. Here, defined as 10 seconds.
- timeout: Maximum time Docker should wait for a health check to complete. Here, defined as 2 seconds.
- retries: Maximum times Docker will try to connect. Here, defined as 5 times.
- start_period: Time to wait before it starts the healthcheck. Here, set to 3 seconds and by default is 0.

**This setup checks whether Postgres is accepting connections. Docker will wait for the check two seconds, will retry up to five times, once every ten seconds, before giving up. If the service responds successfully, Docker will mark the container as “healthy.”**

We can also add a `depends_on` condition to your app service. This ensures the **ETL script won’t even try to start until the database is ready**:

```
services:
  ...
  app:
    build: .
    depends_on:
      db:
        condition: service_healthy
   ...
```

Afterwards, we need to run Docker compose and check for the (healthy).

In [ ]:
docker compose up
docker ps

### Questions

1. You add a health check to your Postgres service in docker-compose.yaml, but you don't update the app service configuration. What will happen when you run docker compose up? 
   The app container may still start before Postgres is ready to accept connections
   Adding a health check alone doesn't change startup order. You need to add depends_on with condition: service_healthy to the app service to ensure it waits for the database to be ready.

2. In the health check configuration, the interval is set to 5s and retries is set to 5. What does this mean for your Postgres container startup?
   Docker will wait up to 25 seconds for Postgres to become healthy before marking it as unhealthy - With 5 retries at 5-second intervals, Docker will attempt the health check up to 5 times over approximately 25 seconds before giving up.

## Optimize Your Dockerfile with Multi-Stage Builds

WHat is multi-stage builds? It is an optimization of the Dockerfile while keeping them easy to follow. Offering a cleaner, more controlled way to produce smaller, production-ready containers.  This technique lets you separate the **build environment** (where you install and compile everything) from the **runtime environment** (the lean final image that actually runs your app). 

Here, you can acces more detailed information about [multi-stage builds](https://docs.docker.com/build/building/multi-stage/).

Now, I am going to explain how to create a two stages Dockerfile, Each stage must be initialize with `FROM` and can present a name with `AS`. Naming a stage makes it easier to use when using `COPY`.

1. Create a stage name 'builder' in which the dependencies are installed as a temporary location. 
   ```
   FROM python:3.10-slim AS builder
   
   WORKDIR /app
   COPY app.py .
   RUN pip install --target=/tmp/deps psycopg2-binary
   ```

2. The final stage starts from a fresh image and copies in only what’s needed to run the app. In this case, we copy the Python libraries and file.
   ```
   FROM python:3.10-slim 
   
   WORKDIR /app
   COPY --from=builder /app/app.py .
   COPY --from=builder /tmp/deps /usr/local/lib/python3.10/site-packages/
   ```

This ensures the final image is small, clean, and free of anything related to development or testing. The Dockerfile will look like:

```
FROM python:3.10-slim AS builder

WORKDIR /app
COPY app.py .
RUN pip install --target=/tmp/deps psycopg2-binary

FROM python:3.10-slim 

WORKDIR /app
COPY --from=builder /app/app.py .
COPY --from=builder /tmp/deps /usr/local/lib/python3.10/site-packages/

CMD ["python", "app.py"]
```
### Image 

During development, using `build`: is often more convenient because you can tweak your Dockerfile and rebuild on the fly. When you're preparing something reproducible for handoff, though, switching to `image`: makes sense because it locks in a specific version of the container.

Rebuild your image using a version tag so it doesn’t overwrite your original and update the information in the Dockerfile as shown here:

```
services:
  app:
    #build: . # Actively developing and want Compose to rebuild the image each time
    image: et-app:v2 # Force Docker Compose to use the tagged image instead of creating a new one (build)
```

This tradeoff is one reason many teams use multiple Compose files:
- A base `docker-compose.yml` for production (using image:)
- A `docker-compose.dev.yml` for local development (with build:)
- A docker-compose.test.yml to replicate CI testing environments

In [ ]:
docker build -t etl-app:v2 .
# Change Dockerfile with image:
docker compose up --build
docker images

Even if your current app is tiny, getting used to multi-stage builds now sets you up for smoother production work later. It separates concerns more clearly, reduces the chance of leaking dev tools into production, and gives you tighter control over what goes into your images.

Some teams even use this structure to compile code in one language and run it in another base image entirely. Others use it to enforce security guidelines by ensuring only tested, minimal files end up in deployable containers.

Whether or not the image size changes much in this case, the structure itself is the win. It gives you portability, predictability, and a cleaner build process without needing to micromanage what’s included.

A single-stage Dockerfile can be tidy on paper, but everything you install or download, even temporarily, ends up in the final image unless you carefully clean it up. Multi-stage builds give you a cleaner separation of concerns by design, which means fewer surprises, fewer manual steps, and less risk of shipping something you didn’t mean to.

### Questions

1. You've created a multi-stage Dockerfile and rebuilt your image with docker build -t etl-app:v2 .. When you run docker compose up, your app still uses the old image. What's the most likely cause?
   Your docker-compose.yml still uses build: . instead of image: etl-app:v2
   Docker Compose rebuilds from the Dockerfile rather than using your tagged image. You need to switch to image: etl-app:v2 to use the specific image you built, or use docker compose up --build to rebuild.
   
2. What is the main advantage of using multi-stage builds over manually removing build tools with RUN rm commands in a single-stage Dockerfile?
   Multi-stage builds start the final image from a clean base, while RUN rm only marks files as deleted but they remain in previous layers
   Docker images are built in layers, and deleting files in a later layer doesn't remove them from earlier layers—they still contribute to the image size. Multi-stage builds avoid this by copying only what you need into a fresh base imag

## Run Your App as a Non-Root User

By default, most containers, including the ones you’ve built so far, run as the root user inside the container. That’s convenient for development, but it’s risky in production. 

Instead of running as root, you’ll create a dedicated user and switch to it before the container runs. In fact, some platforms require non-root users to work properly. Making the switch early can prevent frustrating errors later on, while also improving your security posture.

This creates a minimal user (-m) and tells Docker to use that account when the container runs. If you’ve already refactored your Dockerfile using multi-stage builds, this change goes in the final stage, after dependencies are copied in and right before the CMD.

```
RUN useradd -m etluser
USER etluser
```

In [ ]:
# Check the current user 
docker compose run app whoami
# Print: etluser

One thing to keep in mind is file permissions. If your app writes to mounted volumes or tries to access system paths, switching away from root can lead to permission errors. You likely won’t run into that in this project, but it’s worth knowing where to look if something suddenly breaks after this change.

### Questions

1. You've added RUN useradd -m etluser and USER etluser to your Dockerfile's final stage. After rebuilding, your app crashes with a permission denied error when trying to write to a mounted volume. What's the most likely cause?
   The mounted volume's permissions on the host are still owned by root, and etluser doesn't have write access.
   When you switch from root to a non-root user, existing file permissions don't automatically adjust. The mounted volume on your host may still be owned by root or another user, preventing etluser from writing to it. You'll need to adjust permissions on the host or inside the container.
   
2. Where should you add the USER etluser directive in a multi-stage Dockerfile?
   In the final stage, after copying dependencies and before the CMD instruction
   The USER directive should go in your final runtime stage, after all the setup work is done (copying files, installing dependencies) but before CMD. This way, the application runs as a non-root user, but you can still perform privileged operations during the build if needed

## Externalize Configuration with .env Files

In earlier steps, you may have hardcoded your Postgres credentials and database name directly into your docker-compose.yaml. That works for quick local tests, but in a real project, it’s a security risk.

Storing secrets like usernames and passwords directly in version-controlled files is never safe. Even in private repos, those credentials can easily leak or be accidentally reused. That’s why one of the first steps toward securing your pipeline is externalizing sensitive values into environment variables.

Docker Compose makes this easy by automatically reading from a .env file in your project directory. This is where you store sensitive environment variables like database passwords, without exposing them in your versioned YAML.

But your .env file should never be committed to version control. Instead, add it to your .gitignore file to keep it private. To make your project safe and shareable, create a .env.example file 

Anyone cloning your project can copy that file, rename it to .env, and customize it for their own use, without risking real secrets or overwriting someone else’s setup.

Externalizing secrets this way is one of the simplest and most important steps toward writing secure, production-ready Docker projects. It also lays the foundation for more advanced workflows down the line, like secret injection from CI/CD pipelines or cloud platforms. The more cleanly you separate config and secrets from your code, the easier your project will be to scale, deploy, and share safely.
 
## Optional Concepts: Going Even Further

The features you’ve added so far, health checks, multi-stage builds, non-root users, and .env files, go a long way toward making your pipeline production-ready. But there are a few more Docker and Docker Compose capabilities that are worth knowing, even if you don’t need to implement them right now.

### Resource Constraints

One of those is **resource constraints**. In shared environments, or when testing pipelines in CI, you might want to restrict how much memory or CPU a container can use. Docker Compose supports this through optional fields like `mem_limit` and `cpu_shares`.

### Logging

Another area to consider is **logging**. By default, Docker Compose captures all `stdout` and `stderr` output from each container. \For most pipelines, that’s enough: you can view logs using docker compose logs or see them live in your terminal. 

In production, though, logs are often forwarded to a centralized service, written to a mounted volume, or parsed automatically for errors. Keeping your logs structured and focused (especially if you use Python’s logging module) makes that transition easier later on.

## Kubernetes

Many of the improvements you’ve made in this lesson map directly to concepts in Kubernetes:

- Health checks become readiness and liveness probes
- Non-root users align with container `securityContext` settings
- Environment variables and `.env` files lay the groundwork for using Secrets and ConfigMaps

Even if you’re not deploying to Kubernetes yet, you’re already building the right habits. These are the same tools and patterns that production-grade pipelines depend on.

### Questions

1. You've set mem_limit: 512m in your Docker Compose file and are testing locally on Docker Desktop. Why might this constraint not be enforced as expected?
   Resource limits don't work on Docker Desktop without extra configuration
   The tutorial mentions that resource constraints 'don't work on Docker Desktop without extra configuration.' They become more important in shared environments or when scaling up.

2. When preparing your Docker Compose pipeline for eventual migration to Kubernetes, which statement best describes the relationship between Docker Compose health checks and Kubernetes probes?
   Docker Compose health checks map to both readiness and liveness probes in Kubernetes
   The lesson mentions that 'Health checks become readiness and liveness probes' in Kubernetes. While Docker Compose has a single health check concept, Kubernetes separates this into two types of probes with different purposes.